In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/alaawaleed88/harrypotter/harrypotter.pdf


In [2]:
!pip install -q pymupdf4llm qdrant-client sentence-transformers langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 1.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 9.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 5.3 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 21.5 MB/s eta 0:00:0000:0100:01


**Creating the markdown file**

In [3]:
import pymupdf


def pdf_to_markdown(pdf_path, markdown_path):
    markdown = []
    with pymupdf.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            markdown.append(f"## Page {page_number}\n\n")
            markdown.append(page.get_text())
            markdown.append("\n\n")

    with open(markdown_path, "w", encoding="utf-8") as file:
        file.write("".join(markdown))

    print(f"Markdown file created: {markdown_path}")


pdf_to_markdown(
    "/kaggle/input/datasets/alaawaleed88/harrypotter/harrypotter.pdf",
    "/kaggle/working/output.md",
)

Markdown file created: /kaggle/working/output.md


In [4]:
with open("/kaggle/working/output.md", "r", encoding="utf-8") as f:
    content = f.read()

print(f"Total characters: {len(content)}")
print(content[:2000])  # first ~2000 chars to eyeball structure

Total characters: 6334537
## Page 1



## Page 2



## Page 3



## Page 4



## Page 5



## Page 6

CONTENTS
Harry Potter and the Sorcerer’s Stone
Harry Potter and the Chamber of Secrets
Harry Potter and the Prisoner of Azkaban
Harry Potter and the Goblet of Fire
Harry Potter and the Order of the Phoenix
Harry Potter and the Half-Blood Prince
Harry Potter and the Deathly Hallows


## Page 7



## Page 8



## Page 9

 
FOR JESSICA, WHO LOVES STORIES,
FOR ANNE, WHO LOVED THEM TOO;
AND FOR DI, WHO HEARD THIS ONE FIRST.


## Page 10

 
CONTENTS
ONE
The Boy Who Lived
TWO
The Vanishing Glass
THREE
The Letters from No One
FOUR
The Keeper of the Keys
FIVE
Diagon Alley
SIX
The Journey from Platform Nine and Three-quarters
SEVEN
The Sorting Hat
EIGHT
The Potions Master
NINE
The Midnight Duel
TEN
Halloween
ELEVEN
Quidditch
TWELVE


## Page 11

The Mirror of Erised
THIRTEEN
Nicolas Flamel
FOURTEEN
Norbert the Norwegian Ridgeback
FIFTEEN
The Forbidden Forest
SIXTEEN
Through the Trapdoor
SEVENTEE

**Text preprocessing**

In [5]:
from pathlib import Path
import re

INPUT_FILE = Path("output.md")
OUTPUT_FOLDER = Path("dataset")

BOOK_RANGES = [
    ("Harry Potter and the Sorcerer Stone", 12, 274),
    ("Harry Potter and the Chamber of Secrets", 282, 565),
    ("Harry Potter and the Prisoner of Azkaban", 573, 939),
    ("Harry Potter and the Goblet of Fire", 949, 1560),
    ("Harry Potter and the Order of the Phoenix", 1570, 2406),
    ("Harry Potter and the Half-Blood Prince", 2409, 2964),
    ("Harry Potter and the Deathly Hallows", 2974, 3622),
]


def get_book_name(page_number):
   for book_name, first_page, last_page in BOOK_RANGES:
        if first_page <= page_number <= last_page:
            return book_name

   return None


def clean_text(text):
    text = re.sub(r'-\s+', '', text)               # join broken words
    text = re.sub(r'\s+', ' ', text).strip()       # remove extra spaces
    text = re.sub(r'^\d{1,4}\s+', '', text)        # remove page numbers we already nummerated them
    text = text.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'") # normalize quotation marks
    return text


def split_pages():

    text = INPUT_FILE.read_text(encoding="utf-8")

    pages = re.split(r"^##\s*Page\s+(\d+)\s*$", text, flags=re.MULTILINE)

    OUTPUT_FOLDER.mkdir(exist_ok=True)

    for i in range(1, len(pages), 2):

        page_number = int(pages[i])
        page_content = clean_text(pages[i + 1])
        book_name = get_book_name(page_number)

        if book_name and len(page_content) > 30:  # skip pages with no/near-no book, or near-empty content

            markdown = (
                f"# {book_name}\n\n"
                f"## Page {page_number}\n\n"
                f"{page_content}"
            )

            file_name = f"{book_name} - Page {page_number}.md"
            (OUTPUT_FOLDER / file_name).write_text(markdown, encoding="utf-8")


split_pages()

In [6]:
def extract_page_number(path):
    # filename like "Harry Potter and the Sorcerer Stone - Page 12.md"
    return int(path.stem.split("Page ")[-1])

files = sorted(OUTPUT_FOLDER.glob("*.md"), key=extract_page_number)

sample = files[0]
print(sample.name)
print(sample.read_text(encoding="utf-8"))

Harry Potter and the Sorcerer Stone - Page 12.md
# Harry Potter and the Sorcerer Stone

## Page 12

M CHAPTER ONE THE BOY WHO LIVED r. and Mrs. Dursley, of number four, Privet Drive, were proud to say that they were perfectly normal, thank you very much. They were the last people you'd expect to be involved in anything strange or mysterious, because they just didn't hold with such nonsense. Mr. Dursley was the director of a firm called Grunnings, which made drills. He was a big, beefy man with hardly any neck, although he did have a very large mustache. Mrs. Dursley was thin and blonde and had nearly twice the usual amount of neck, which came in very useful as she spent so much of her time craning over garden fences, spying on the neighbors. The Dursleys had a small son called Dudley and in their opinion there was no finer boy anywhere. The Dursleys had everything they wanted, but they also had a secret, and their greatest fear was that somebody would discover it. They didn't think the

**Creating Embeddings**

In [7]:
from sentence_transformers import SentenceTransformer
from pathlib import Path
import torch

DATASET_FOLDER = Path("/kaggle/working/dataset")
MODEL_NAME = "intfloat/multilingual-e5-small"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def extract_page_number(file_path):
    return int(file_path.stem.split("Page ")[-1])


def read_page(file_path):
    lines = file_path.read_text(encoding="utf-8").splitlines()

    book_name = lines[0].replace("# ", "")
    page_number = int(lines[2].replace("## Page ", ""))
    content = " ".join(lines[3:]).strip()

    return {
        "book_name": book_name,
        "page_number": page_number,
        "content": content,
    }


files = sorted(DATASET_FOLDER.glob("*.md"), key=extract_page_number)
pages = [read_page(file) for file in files]
texts = [f"passage: {page['content']}" for page in pages]

print(f"Total pages to embed: {len(pages)}")
print(f"Using device: {DEVICE}")

model = SentenceTransformer(MODEL_NAME, device=DEVICE)

embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=32,
).tolist()

print(f"Embedding dimension: {len(embeddings[0])}")

Total pages to embed: 3564
Using device: cuda


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/112 [00:00<?, ?it/s]

Embedding dimension: 384


In [11]:
from qdrant_client import QdrantClient, models
from kaggle_secrets import UserSecretsClient


user_secrets = UserSecretsClient()
QDRANT_URL = user_secrets.get_secret("QDRANT_URL")
QDRANT_API_KEY = user_secrets.get_secret("QDRANT_API_KEY")
QDRANT_COLLECTION = user_secrets.get_secret("QDRANT_COLLECTION")

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)

collection_name = QDRANT_COLLECTION
vector_size = len(embeddings[0])

if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=vector_size,
            distance=models.Distance.COSINE,
        ),
    )
    print(f"Created collection '{collection_name}' with vector size {vector_size}")
else:
    print(f"Collection '{collection_name}' already exists")

points = [
    models.PointStruct(
        id=index,
        vector=embedding,
        payload=page,
    )
    for index, (page, embedding) in enumerate(zip(pages, embeddings))
]

batch_size = 100
for start in range(0, len(points), batch_size):
    batch = points[start:start + batch_size]
    client.upsert(
        collection_name=collection_name,
        points=batch,
    )
    print(f"Uploaded {start + len(batch)}/{len(points)}")

print(f"✅ Uploaded {len(points)} pages to Qdrant collection '{collection_name}'.")

Created collection 'HarryPotter_Rag_Chatbot' with vector size 384
Uploaded 100/3564
Uploaded 200/3564
Uploaded 300/3564
Uploaded 400/3564
Uploaded 500/3564
Uploaded 600/3564
Uploaded 700/3564
Uploaded 800/3564
Uploaded 900/3564
Uploaded 1000/3564
Uploaded 1100/3564
Uploaded 1200/3564
Uploaded 1300/3564
Uploaded 1400/3564
Uploaded 1500/3564
Uploaded 1600/3564
Uploaded 1700/3564
Uploaded 1800/3564
Uploaded 1900/3564
Uploaded 2000/3564
Uploaded 2100/3564
Uploaded 2200/3564
Uploaded 2300/3564
Uploaded 2400/3564
Uploaded 2500/3564
Uploaded 2600/3564
Uploaded 2700/3564
Uploaded 2800/3564
Uploaded 2900/3564
Uploaded 3000/3564
Uploaded 3100/3564
Uploaded 3200/3564
Uploaded 3300/3564
Uploaded 3400/3564
Uploaded 3500/3564
Uploaded 3564/3564
✅ Uploaded 3564 pages to Qdrant collection 'HarryPotter_Rag_Chatbot'.
